# SubClassing 모델 사용 - 데이터 섞기, GradientTape

In [26]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [27]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 구조 변경(차원)
print(x_train.shape)    # (60000,, 28, 28)
x_train = x_train.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
x_test = x_test.reshape((-1, 28, 28, 1)).astype('float32') / 255.0
print(x_train.shape)    # (60000, 28, 28, 1)

(60000, 28, 28)
(60000, 28, 28, 1)


# tf.data.Dataset.from_tensor_slices 사용하기

In [28]:
# tf.data.Dataset.from_tensor_slices 
import numpy as np
x = np.random.sample((5, 2))
print(x)

# dset = tf.data.Dataset.from_tensor_slices(x)
# print(dset)
dset = tf.data.Dataset.from_tensor_slices(x).shuffle(10000).batch(1)
print(dset)
for a in dset:
    print(a)

[[0.91571813 0.56332511]
 [0.96696341 0.88806015]
 [0.2496248  0.9403639 ]
 [0.95485772 0.36021186]
 [0.2786669  0.43541614]]
<BatchDataset element_spec=TensorSpec(shape=(None, 2), dtype=tf.float64, name=None)>
tf.Tensor([[0.96696341 0.88806015]], shape=(1, 2), dtype=float64)
tf.Tensor([[0.91571813 0.56332511]], shape=(1, 2), dtype=float64)
tf.Tensor([[0.2496248 0.9403639]], shape=(1, 2), dtype=float64)
tf.Tensor([[0.2786669  0.43541614]], shape=(1, 2), dtype=float64)
tf.Tensor([[0.95485772 0.36021186]], shape=(1, 2), dtype=float64)


In [29]:
# train data 섞기
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(60000).batch(32)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(32)
print(train_ds)
print(test_ds)

<BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>
<BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>


In [30]:
# model
from numpy import dtype
import test


class MyModel(tf.keras.Model):
    def __init__(self):
        super(MyModel, self).__init__()

        self.conv1 = tf.keras.layers.Conv2D(filters=32, kernel_size=[3,3], padding='valid', activation='relu')
        self.pool1 = tf.keras.layers.MaxPool2D(pool_size=(2, 2))

        self.conv2 = tf.keras.layers.Conv2D(filters=32, kernel_size=[3,3], padding='valid', activation='relu')
        self.pool2 = tf.keras.layers.MaxPool2D(pool_size=(2, 2))

        self.flatten = tf.keras.layers.Flatten(dtype='float32')

        self.d1 = tf.keras.layers.Dense(32, activation='relu')
        self.drop1 = tf.keras.layers.Dropout(rate =0.3)
        self.d2 = tf.keras.layers.Dense(10, activation='softmax')
    
    def call(self, inputs):
        net = self.conv1(inputs)
        net = self.pool1(net)
        net = self.conv2(net)
        net = self.pool2(net)
        net = self.flatten(net)
        net = self.d1(net)
        net = self.drop1(net)
        net = self.d2(net)
        return net

model = MyModel()
temp_inputs = tf.keras.Input(shape=(28, 28, 1))
model(temp_inputs)

loss_object = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam()

'''  가장 일반적인 방법
model.compile(optimizer=optimizer , loss=loss_object, metrics=['acc'])
model.fit(x_train, y_train, batch_size=128, epochs=5, verbose=2)    # process 기반 스레딩 처리
score = model.evaluate(x_test, y_test)
print('test loss: ', score[0])
print('test acc: ', score[1])
print('예측값: ', np.argmax(model.predict(x_test[:2]), 1))
print('실제값: ', y_test[:2])
'''

"  가장 일반적인 방법\nmodel.compile(optimizer=optimizer , loss=loss_object, metrics=['acc'])\nmodel.fit(x_train, y_train, batch_size=128, epochs=5, verbose=2)    # process 기반 스레딩 처리\nscore = model.evaluate(x_test, y_test)\nprint('test loss: ', score[0])\nprint('test acc: ', score[1])\nprint('예측값: ', np.argmax(model.predict(x_test[:2]), 1))\nprint('실제값: ', y_test[:2])\n"

# GradientTape을 운영 - 모델 서브 프로세싱 학습방법
모델 손실과 성능을 측정할 지표 선택. 수집된 측정 지표를 바탕으로 최종 결과 출력을 위한 객체 생성

In [33]:
from calendar import EPOCH
from re import template
from numpy import gradient


train_loss = tf.keras.metrics.Mean()    # 주어진 값의 (가중)평균을 계산
train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()
test_loss = tf.keras.metrics.Mean()  
test_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()

@tf.function
def train_step(images, labels): # 얘를 반복하면 loss를 최소화
    with tf.GradientTape() as tape:
        predictions = model(images)
        loss = loss_object(labels, predictions)
    
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_loss(loss)
    train_accuracy(labels, predictions)

    

@tf.function
def test_step(images, labels):
    predictions = model(images)
    t_loss = loss_object(labels, predictions)
    test_loss(t_loss)
    test_accuracy(labels, predictions)

EPOCHS = 5
for epoch in range(EPOCHS):
    for train_images, train_labels in train_ds:
        train_step(train_images, train_labels)

    for test_images, test_labels in test_ds:
        test_step(test_images, test_labels)
    
    template = 'epochs:{}, train_loss:{}, train_acc:{}, test_loss:{}, test_acc:{}'
    print(template.format(epoch + 1, train_loss.result(), train_accuracy.result() * 100, test_loss.result(), test_accuracy.result() * 100))
    



epochs:1, train_loss:0.04551079869270325, train_acc:98.57500457763672, test_loss:0.042905230075120926, test_acc:98.58999633789062
epochs:2, train_loss:0.040223680436611176, train_acc:98.73750305175781, test_loss:0.04075122997164726, test_acc:98.68000030517578
epochs:3, train_loss:0.03618011251091957, train_acc:98.86055755615234, test_loss:0.03916342556476593, test_acc:98.76667022705078
epochs:4, train_loss:0.03306041285395622, train_acc:98.95916748046875, test_loss:0.038428183645009995, test_acc:98.79000091552734
epochs:5, train_loss:0.03030865266919136, train_acc:99.03766632080078, test_loss:0.03883897140622139, test_acc:98.802001953125


In [34]:
print('예측값: ', np.argmax(model.predict(x_test[:2]), 1))
print('실제값: ', y_test[:2])

1/1 [==============================] - 0s 57ms/step
예측값:  [7 2]
실제값:  [7 2]
